In [ ]:
import sys, pathlib
_root = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (c / "projects" / "data").is_dir()), pathlib.Path.cwd())
sys.path.insert(0, str(_root))
from projects.data.make_synth_table import make_messy_table


# 04 · 缺失值与数据类型  Missing Data & dtypes

> **能力标签**：SH4（数据处理与分析）

## 目标 Objectives

完成本课后，你应该能够：

1. 用 `isna()` / `notna()` 检测缺失值（NaN / None / NaT）。
2. 用 `fillna()`、`dropna()` 处理缺失值，并理解各参数的含义。
3. 用 `astype()`、`pd.to_numeric()`、`pd.to_datetime()` 正确转换列类型。
4. 用 `drop_duplicates()` 去除重复行，理解 `keep` 参数。
5. 实现 `clean(df)` —— W3-clean 题包核心函数，并理解为何要用 `.copy()` 避免 SettingWithCopyWarning。

> 对应能力：**SH4**


## 概念讲解 Concepts

### 缺失值检测

pandas 用 `np.nan`（浮点 NaN）和 `pd.NaT`（时间类型缺失）表示缺失值：

```python
df.isna()           # 返回同形状 bool DataFrame
df.isna().sum()     # 每列缺失值数量
df["x1"].notna()   # 非缺失掩码
```

---

### 填补与丢弃

```python
df.fillna(0)                           # 全部填 0
df["x1"].fillna(df["x1"].median())   # 用中位数填
df.dropna()                            # 丢弃含 NaN 的行
df.dropna(subset=["x1"])              # 只看 x1 列
df.dropna(thresh=3)                    # 至少 3 个非 NaN 才保留
```

---

### dtype 转换

```python
df["age"] = df["age"].astype(int)
df["price"] = pd.to_numeric(df["price"], errors="coerce")  # 无法转换→NaN
df["date"] = pd.to_datetime(df["date"])
```

---

### 去重

```python
df.drop_duplicates()                        # 保留第一次出现的行
df.drop_duplicates(keep="last")             # 保留最后一次
df.drop_duplicates(subset=["record_id"])   # 只看特定列
```

---

### View vs Copy 与 SettingWithCopyWarning

> **重要**：对 DataFrame 切片再赋值时，pandas 不确定结果是视图（view）还是副本（copy），
> 可能触发 `SettingWithCopyWarning`。

正确做法：先 `.copy()` 创建独立副本再修改：

```python
out = df.drop_duplicates().copy()    # 明确复制
out["group"] = out["group"].str.upper()  # 安全修改
```

---

### `clean` 函数签名（题包核心）

```python
def clean(df):
    out = df.drop_duplicates().copy()
    for col in out.select_dtypes(include='number').columns:
        out[col] = out[col].fillna(out[col].median())
    if 'group' in out.columns:
        out['group'] = out['group'].astype(str).str.upper()
    return out.reset_index(drop=True)
```

## 示例 Worked Example

In [ ]:
import pandas as pd
import numpy as np
from projects.data.make_synth_table import make_messy_table

# 使用内置的脏数据（含 NaN、大小写混用 group、重复行）
df_messy = make_messy_table(n=100, seed=42)
print('=== 合成表格数据前 5 行 ===')
print(df_messy.head())

# 1. 检测缺失值
print('\n=== 缺失值计数 ===')
print(df_messy.isna().sum())

# 2. fillna：数值列用中位数
print('\n=== 填补前 x1 空值数:', df_messy['x1'].isna().sum(), '===')
df_filled = df_messy.copy()
for col in df_filled.select_dtypes(include='number').columns:
    df_filled[col] = df_filled[col].fillna(df_filled[col].median())
print('填补后 x1 空值数:', df_filled['x1'].isna().sum())

# 3. drop_duplicates
print('\n=== 去重前行数:', len(df_messy), '===')
df_deduped = df_messy.drop_duplicates()
print('去重后行数:', len(df_deduped))

# 4. astype 示例
df_deduped = df_deduped.copy()
df_deduped['record_id'] = df_deduped['record_id'].astype(int)
print('\nrecord_id dtype:', df_deduped['record_id'].dtype)


## 动手 Exercise

在下面的 cell 中：

1. 用 `make_messy_table(n=200, seed=7)` 获取脏数据。
2. 打印各列缺失值数量。
3. 实现完整的 `clean` 函数（去重 → 数值中位数填补 → group 大写），并应用到数据上。
4. 验证清洗后 `isna().sum()` 全为 0，`group` 只含 `'A'` / `'B'`。


In [ ]:
import pandas as pd
from projects.data.make_synth_table import make_messy_table

df = make_messy_table(n=200, seed=7)

# 1. 缺失值计数
print('=== 缺失值 ===')
print(df.isna().sum())

# 2. 实现 clean
def clean(df):
    out = df.drop_duplicates().copy()
    for col in out.select_dtypes(include='number').columns:
        out[col] = out[col].fillna(out[col].median())
    if 'group' in out.columns:
        out['group'] = out['group'].astype(str).str.upper()
    return out.reset_index(drop=True)

df_clean = clean(df)

# 3. 验证
print('\n=== 清洗后缺失值 ===')
print(df_clean.isna().sum())
print('\n=== group 值分布 ===')
print(df_clean['group'].value_counts())


## 延伸阅读 Further Reading

1. **pandas 官方文档 — 缺失数据**: <https://pandas.pydata.org/docs/user_guide/missing_data.html>
2. **pandas 官方文档 — dtypes**: <https://pandas.pydata.org/docs/user_guide/basics.html#dtypes>
3. **SettingWithCopyWarning 详解**: <https://pandas.pydata.org/docs/user_guide/indexing.html#returning-a-view-versus-a-copy>
4. **《Python for Data Analysis》第 7 章** — Wes McKinney


## 关联练习 Related Assignments

本课对应 **W3-clean** 题包，核心函数为 `clean`：

```bash
python check.py w3-clean
```

> 能力标签：**SH4** · threshold ≥ 0.7
